# # Load training data

In [1]:
# Demo scaffolding — replace with your own data source in production.
from src.iris_demo import prepare
conn_name = prepare()

# Reuse a JupyterLab SQL Explorer connection by name (no host/credentials
# in the notebook). `data.split` propagates source_uri lineage to each subset.
#   data.connection(name)   → bound DB connection object
#   db.load(query)          → DataFrame
#   data.split(df, ratios, random_state=...) → {name: subset} dict
from nubison_model import data

db = data.connection(conn_name)
df = db.load("SELECT * FROM iris")
datasets = data.split(df, {"training": 0.8, "evaluation": 0.2}, random_state=42)

for name, sub in datasets.items():
    print(f"{name:12s} rows={len(sub):3d}  source_uri={sub.attrs['source_uri']}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


training     rows=120  source_uri=dbref://IRIS#a295df6f/training

evaluation   rows= 30  source_uri=dbref://IRIS#a295df6f/evaluation

# # Train (sklearn / xgboost / lightgbm / keras)

In [2]:
# `train()` opens an MLflow run with autolog + dataset lineage and yields
# a `TrainContext`. `t.fit(estimator)` runs `estimator.fit(t.X, t.y)`, pickles
# the fitted model to `weights_path`, and logs an `evaluation_score`.
#   datasets     — dict from `data.split` (must include "training")
#   target       — label column name (or list of names)
#   weights_path — where to save the trained model
#
# Works for any estimator with a sklearn `fit(X, y, **kwargs)` API:
# sklearn / xgboost / lightgbm / keras / skorch.
from nubison_model import train
from sklearn.linear_model import LogisticRegression

WEIGHTS_PATH = "src/weights.pkl"

with train(datasets=datasets, target="target", weights_path=WEIGHTS_PATH) as t:
    t.fit(LogisticRegression(max_iter=100))

print(f"run_id: {t.run_id}")
print(f"weights: {WEIGHTS_PATH}")


2026/05/15 18:32:12 INFO mlflow.tracking.fluent: Experiment with name 'MultiFW_1778837529' does not exist. Creating a new experiment.


2026/05/15 18:32:12 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


2026/05/15 18:32:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run mercurial-croc-778 at: http://127.0.0.1:5050/#/experiments/5/runs/ef25292414b344e9bf90f9d168764902


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/5


run_id: ef25292414b344e9bf90f9d168764902

weights: src/weights.pkl

# # Next: package for inference

The trained estimator is now pickled at `src/weights.pkl`. To serve it as a
REST endpoint, see `model.ipynb` in this same directory.